In [1]:
from main_experiment_qpp_perplexity import Main_Experiment

In [2]:
lab = Main_Experiment()

In [6]:
lab.experiment('e5', 3)

10:24:26.230 [main] WARN org.terrier.structures.BaseCompressingMetaIndex -- Structure meta reading data file directly from disk (SLOW) - try index.meta.data-source=fileinmem in the index properties file. 1.9 GiB of memory would be required.


In [4]:
import pandas as pd

x = pd.read_csv(f'./precomputed_qpps/e5_3_combined_qpp_dl_test.csv')
x

,qid,query,qpp_estimate,parameters,qpp_method
0,1030303,who is aziz hashim,0.885742,{'position': 0},maxScore
1,1037496,who is rep scalise,0.888616,{'position': 0},maxScore
2,1037798,who is robert gray,0.860324,{'position': 0},maxScore
3,1043135,who killed nicholas ii of russia,0.890066,{'position': 0},maxScore
4,104861,cost of interior concrete flooring,0.906576,{'position': 0},maxScore
...,...,...,...,...,...
480,911232,what type of conflict does della face in o hen...,0.295146,"{'encoder': 'bert-base-uncased', 'k': 1}",bertQPP(QV)
481,914916,what type of tissue are bronchioles,0.058931,"{'encoder': 'bert-base-uncased', 'k': 1}",bertQPP(QV)
482,938400,when did family feud come out,0.121425,"{'encoder': 'bert-base-uncased', 'k': 1}",bertQPP(QV)
483,940547,when did rock n roll begin,0.077644,"{'encoder': 'bert-base-uncased', 'k': 1}",bertQPP(QV)


In [5]:
# for the convenience of analysis, we can do this transformation

transformed_qpp_res = x[['qid', 'query']].drop_duplicates().copy()

for qpp_name in x.qpp_method.unique():
    value_dict = dict(zip(x[x.qpp_method==qpp_name]['qid'], x[x.qpp_method==qpp_name]['qpp_estimate']))
    transformed_qpp_res[qpp_name] = transformed_qpp_res.qid.apply(lambda _qid: value_dict[_qid])

transformed_qpp_res

,qid,query,maxScore,spatial,a_ratio,bertQPP,bertQPP(QV)
0,1030303,who is aziz hashim,0.885742,2912.018066,1.099638,0.765038,0.770591
1,1037496,who is rep scalise,0.888616,2932.444092,1.075504,0.457166,0.214858
2,1037798,who is robert gray,0.860324,2709.392578,1.045476,0.319397,0.594975
3,1043135,who killed nicholas ii of russia,0.890066,2840.372070,1.014932,0.191483,0.221473
4,104861,cost of interior concrete flooring,0.906576,3027.519775,1.016350,0.031656,0.041708
...,...,...,...,...,...,...,...
92,915593,what types of food can you cook sous vide,0.928994,3122.982666,1.031612,0.447705,0.390066
93,938400,when did family feud come out,0.904730,3099.095703,1.055543,0.083752,0.121425
94,940547,when did rock n roll begin,0.892663,2956.253174,1.058689,0.153896,0.077644
95,962179,when was the salvation army founded,0.900481,3120.728760,1.090779,0.430263,0.527964


In [6]:
'spatial' not in x.qpp_method.unique()

False

In [7]:
res = pd.read_csv(f'../../rag_utility/res/e5_dl_19.csv')

In [39]:
grouped = res.groupby('qid').apply(lambda _group_res: _group_res[_group_res['rank']==0])
grouped = grouped.reset_index(drop=True)
grouped = grouped.rename(columns={'score': 'qpp_estimate'})[['qid', 'query', 'qpp_estimate']]
grouped['parameters'] = np.full(grouped.shape[0], {'position': 0})
grouped['qpp_method'] = 'maxScore'
x = pd.concat([x, grouped])
x

,qid,query,qpp_estimate,parameters,qpp_method
0,1030303,who is aziz hashim,2.005307e-03,{'k': 3},nqc
1,1037496,who is rep scalise,5.561743e-07,{'k': 3},nqc
2,1037798,who is robert gray,7.672339e-04,{'k': 3},nqc
3,1043135,who killed nicholas ii of russia,8.886868e-04,{'k': 3},nqc
4,104861,cost of interior concrete flooring,3.601774e-05,{'k': 3},nqc
...,...,...,...,...,...
38,1121402,what can contour plowing reduce,9.195973e-01,{'position': 0},maxScore
39,1121709,what are the three percenters,8.957261e-01,{'position': 0},maxScore
40,1124210,tracheids are part of,8.951566e-01,{'position': 0},maxScore
41,1129237,hydrogen is a liquid below what temperature,9.171942e-01,{'position': 0},maxScore


In [9]:
x.head()

,qid,query,qpp_estimate,parameters,qpp_method
0,1030303,who is aziz hashim,2.005307e-03,{'k': 3},nqc
1,1037496,who is rep scalise,5.561743e-07,{'k': 3},nqc
2,1037798,who is robert gray,7.672339e-04,{'k': 3},nqc
3,1043135,who killed nicholas ii of russia,8.886868e-04,{'k': 3},nqc
4,104861,cost of interior concrete flooring,3.601774e-05,{'k': 3},nqc


In [10]:
# for i in res.groupby(['qid']):
#     i.sort_values(by=['score'], ascending=False)

In [46]:
import numpy as np

def nqc(group, _k=100):
    group = group.sort_values(by=['score'], ascending=False)
    group = group.iloc[:_k]
    scores_k = np.array(group.score.values)
    var_value = np.var(scores_k)

    # queries = group[['qid', 'query']].drop_duplicates()
    # return queries
    return var_value

def max_score(group):
    top_value = group[group['rank']==0].score.values[0]
    return top_value

In [47]:
res.groupby(['qid']).apply(lambda x: nqc(x)).reset_index(drop=True)

0     0.000063
1     0.000220
2     0.000096
3     0.000167
4     0.000022
5     0.000649
6     0.000284
7     0.000403
8     0.000103
9     0.000138
10    0.000066
11    0.000317
12    0.000218
13    0.000192
14    0.000090
15    0.000426
16    0.000276
17    0.000141
18    0.000084
19    0.000018
20    0.000275
21    0.000112
22    0.000408
23    0.000381
24    0.000101
25    0.000124
26    0.000508
27    0.000111
28    0.000130
29    0.000262
30    0.000419
31    0.000111
32    0.000154
33    0.000052
34    0.000173
35    0.000131
36    0.000256
37    0.000115
38    0.000822
39    0.000399
40    0.000067
41    0.000251
42    0.000136
dtype: float64

In [45]:
res.groupby(['qid']).apply(lambda x: max_score(x)).reset_index(drop=True)

0     0.897398
1     0.921980
2     0.923386
3     0.902709
4     0.906576
5     0.938813
6     0.908132
7     0.926675
8     0.922615
9     0.909955
10    0.925398
11    0.897602
12    0.913001
13    0.899960
14    0.936129
15    0.925401
16    0.914295
17    0.900281
18    0.887009
19    0.899789
20    0.902133
21    0.935655
22    0.940348
23    0.895237
24    0.891944
25    0.928994
26    0.900481
27    0.860324
28    0.888884
29    0.882555
30    0.927114
31    0.907097
32    0.879508
33    0.860989
34    0.910462
35    0.919648
36    0.895102
37    0.887165
38    0.919597
39    0.895726
40    0.895157
41    0.917194
42    0.913290
dtype: float64